# TasNet Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
import torchaudio
import numpy as np
import os

In [2]:
class SpeechDataset(Dataset):
  def __init__(self, mix_dir, s1_dir, s2_dir, sample_rate=16000, num_samples=32000):
    self.mix_dir = mix_dir
    self.s1_dir = s1_dir
    self.s2_dir = s2_dir
    self.sample_rate = sample_rate
    self.num_samples = num_samples

    # Assuming all files have the same name for mix, s1, and s2
    self.ids = [os.path.splitext(f)[0] for f in os.listdir(mix_dir) if f.endswith('.wav')]

  def __len__(self):
    return len(self.ids)

  def _load_audio(self, path):
    wav, sr = torchaudio.load(path)
    wav = wav.mean(0)  # Convert to mono if necessary
    if sr != self.sample_rate:
        wav = torchaudio.functional.resample(wav, sr, self.sample_rate)
    if wav.shape[0] < self.num_samples:
        wav = F.pad(wav, (0, self.num_samples - wav.shape[0]))
    else:
        wav = wav[:self.num_samples]
    return wav

  def __getitem__(self, idx):
    utt_id = self.ids[idx]
    mix_wav = self._load_audio(os.path.join(self.mix_dir, utt_id + '.wav'))
    s1_wav  = self._load_audio(os.path.join(self.s1_dir, utt_id + '.wav'))
    s2_wav  = self._load_audio(os.path.join(self.s2_dir, utt_id + '.wav'))
    sources = torch.stack([s1_wav, s2_wav], dim=0)  # (2, T)
    return mix_wav, sources

<img src="./Images/hyperparameters.png" width=480 />

In [33]:
class Conv1DBlock(nn.Module):
  def __init__(self, in_ch, hidden_ch, kernel_size, dilation):
    super().__init__()
    self.conv1 = nn.Conv1d(in_ch, hidden_ch, 1)
    self.prelu1 = nn.PReLU()
    self.norm1 = nn.GroupNorm(1, hidden_ch)
    self.dwconv = nn.Conv1d(hidden_ch, hidden_ch, kernel_size,
                            padding=(kernel_size-1)//2 * dilation,
                            dilation=dilation, groups=hidden_ch)
    self.prelu2 = nn.PReLU()
    self.norm2 = nn.GroupNorm(1, hidden_ch)
    self.resconv = nn.Conv1d(hidden_ch, in_ch, 1)

  def forward(self, x):
    out = self.conv1(x)
    out = self.prelu1(out)
    out = self.norm1(out)
    out = self.dwconv(out)
    out = self.prelu2(out)
    out = self.norm2(out)
    out = self.resconv(out)
    return out + x

class Separator(nn.Module):
  def __init__(self, N=256, B=256, H=512, P=3, X=6, R=2, num_spks=2):
    super().__init__()
    self.layernorm = nn.GroupNorm(1, N)
    self.bottleneck = nn.Conv1d(N, B, 1)
    self.TCN = nn.ModuleList()
    for r in range(R):
      for x in range(X):
        self.TCN.append(Conv1DBlock(B, H, P, dilation=2**x))
    self.mask_conv = nn.Conv1d(B, num_spks * N, 1)
    self.N = N
    self.num_spks = num_spks

  def forward(self, x):
    x = self.layernorm(x)
    x = self.bottleneck(x)
    for block in self.TCN:
      x = block(x)
    masks = self.mask_conv(x)
    B, _, T = masks.shape
    masks = masks.view(B, self.num_spks, self.N, T)
    masks = F.relu(masks)
    return masks

class ConvTasNet(nn.Module):
  def __init__(self, N=256, L=20, **kwargs):
    super().__init__()
    self.encoder = nn.Conv1d(1, N, L, stride=L // 2, bias=False)
    self.separator = Separator(N=N, **kwargs)
    self.decoder = nn.ConvTranspose1d(N, 1, L, stride=L // 2, bias=False)

  def forward(self, mixture):
    x = mixture.unsqueeze(1)  # (B, 1, T)
    enc = self.encoder(x)     # (B, N, T')
    print(enc.shape)
    masks = self.separator(enc)  # (B, C, N, T')
    print(masks.shape)
    masked = masks * enc.unsqueeze(1)  # (B, C, N, T')
    B, C, N, T = masked.shape
    masked = masked.view(B*C, N, T)
    decoded = self.decoder(masked)  # (B*C, 1, T_out)
    decoded = decoded.squeeze(1)
    decoded = decoded.view(B, C, -1)  # (B, C, T_out)
    return decoded

In [ ]:
mix_dir = "./Dataset/train-100/mix_clean"
s1_dir = "./Dataset/train-100/s1"
s2_dir = "./Dataset/train-100/s2"

dataset = SpeechDataset(
  mix_dir,
  s1_dir,
  s2_dir,
  sample_rate=8000,
  num_samples=40000
)

small_dataset = Subset(
  dataset,
  range(2000)
)

dataloader = DataLoader(
  small_dataset,
  batch_size=4,
  shuffle=True
)

In [35]:
x = torch.randn(1, 40000)
testconnet = ConvTasNet()
testout = testconnet(x)

torch.Size([1, 256, 3999])
torch.Size([1, 2, 256, 3999])


In [34]:
mix, sources = next(iter(dataloader))

print(mix.shape)  # (8, 40000)
print(sources.shape) # (8, 2, 40000)

torch.Size([4, 40000])
torch.Size([4, 2, 40000])


In [ ]:
def si_snr(est, ref, eps=1e-8):
  ref = ref - ref.mean(dim=1, keepdim=True)
  est = est - est.mean(dim=1, keepdim=True)
  s_target = (torch.sum(est * ref, dim=1, keepdim=True) * ref) / (
      torch.sum(ref ** 2, dim=1, keepdim=True) + eps)
  e_noise = est - s_target
  si_snr_val = 10 * torch.log10(
      (torch.sum(s_target ** 2, dim=1) + eps) /
      (torch.sum(e_noise ** 2, dim=1) + eps))
  return si_snr_val

def pit_loss(ests, refs):
  # ests/refs: (B, 2, T)
  loss1 = si_snr(ests[:, 0], refs[:, 0]) + si_snr(ests[:, 1], refs[:, 1])
  loss2 = si_snr(ests[:, 0], refs[:, 1]) + si_snr(ests[:, 1], refs[:, 0])
  # maximize SI-SNR, so minimize negative SI-SNR
  return -torch.mean(torch.stack([loss1, loss2], dim=0).max(dim=0)[0]) / 2

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ConvTasNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 10

for epoch in range(num_epochs):
  model.train()
  running_loss = 0

  print(f" Epoch: {epoch+1}/{num_epochs}")
  i = 0
  for mix, sources in dataloader:
    mix = mix.to(device)
    sources = sources.to(device)

    est_sources = model(mix)   # (B, 2, T)
    min_len = min(est_sources.shape[-1], sources.shape[-1])
    est_sources = est_sources[:, :min_len]
    sources = sources[:, :min_len]
    
    loss = pit_loss(est_sources, sources)
    print(f"{i + 1}: {loss}")
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    running_loss += loss.item()
    i += 1
  print(f'Average Loss: {running_loss / len(dataloader):.4f}')

 Epoch: 1/10
1: 27.491758346557617
2: 8.24778938293457
3: 4.6057610511779785
4: 3.1559882164001465
5: 3.5732834339141846
6: 2.4098477363586426
7: 2.255436420440674
8: 2.0790131092071533
9: 1.7960795164108276
10: 1.5993307828903198
11: 1.2150585651397705
12: 0.614484429359436
13: 1.306044578552246
14: 2.454819679260254
15: 0.8478864431381226
16: 1.9831700325012207
17: 1.1592390537261963
18: 2.064565658569336
19: 1.1022086143493652
20: 1.5698528289794922
21: 0.4579440951347351
22: 0.6552680730819702
23: 0.24137024581432343
24: 1.191933035850525
25: 1.2478091716766357
26: 1.2559583187103271
27: -0.09903895854949951
28: 0.7983521819114685
29: 0.7806071043014526
30: 1.1351717710494995
31: 0.532418966293335
32: 0.779300332069397
33: 0.5223482847213745
34: 0.6264344453811646
35: 0.30069923400878906
36: 0.8814812302589417
37: 0.3556710481643677
38: 0.42380809783935547
39: 0.8139182925224304
40: 0.7380679249763489
41: 1.0555099248886108
42: -0.5402262210845947
43: 0.26191458106040955
44: 0.4821

In [14]:
import IPython.display as ipd

model.eval()
with torch.no_grad():
  mix, sources = dataset[4000]
  est_sources = model(mix.unsqueeze(0).to(device)).cpu().squeeze(0)
  print("Mixture")
  ipd.display(ipd.Audio(mix.numpy(), rate=8000))
  print("Estimated Speaker 1")
  ipd.display(ipd.Audio(est_sources[0].numpy(), rate=8000))
  print("Estimated Speaker 2")
  ipd.display(ipd.Audio(est_sources[1].numpy(), rate=8000))

Mixture


Estimated Speaker 1


Estimated Speaker 2
